# Phase 1: routing data collection (Colab)

Collects Phase 1 routing telemetry from Qwen1.5-MoE-A2.7B (fp16) and validates it.

**What it does:**
1. Checks for a GPU, clones the repo at the chosen branch or commit, installs the pinned requirements, and confirms the installed versions match.
2. Collects **every prompt once** (GSM8K train + test, all MBPP splits: about 9,800 prompts) into `full/`, plus a stratified **200-prompt subset per dataset in 5 orderings** into `determinism/` (#109).
3. Validates the data, runs the batch-variance analysis, and writes `provenance.json` (commit, GPU, versions, times, counts).
4. Copies everything to a Drive folder named after the commit.

**Runtime:** about 6 minutes of collection on an A100 (measured at `38ef1da`), plus a few minutes of setup and validation, plus the copy to Drive (about 3.3 GB).

**Before you start:** Runtime → Change runtime type → **A100 GPU**. Colab may give you the 40 GB or the 80 GB A100; `provenance.json` records which.

**When you're done:** disconnect the runtime (last section). Colab bills for GPU time as long as it's connected.

## Settings

In [ ]:
# Branch, tag or commit to collect from. The official collection uses "main", after
# the Phase 1 PRs are reviewed and merged; until then, the testing branch.
REF = "integration/phase-1-validation"

NUM_ORDERINGS = 5      # orderings of the determinism subset (#109)
SUBSET_SIZE = None     # None = every prompt. A number limits prompts per dataset, for a quick test run.
COPY_TO_DRIVE = True   # results are written to local disk first, then copied here:
DRIVE_FOLDER = "/content/drive/My Drive/moe_phase1_data"

## Setup

In [ ]:
# Fail fast if there's no GPU, before cloning or installing anything
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU. Runtime → Change runtime type → A100 GPU, then run again."
print("GPU:", gpu.stdout.strip())

In [ ]:
# Mount Drive now, so the sign-in prompt doesn't block an unattended run at the end
if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Clone the repo at REF and install the pinned requirements
REPO = "/content/moe-expert-prefetching"
!rm -rf {REPO}
!git clone --quiet https://github.com/scerruti/moe-expert-prefetching.git {REPO}
!git -C {REPO} checkout --quiet {REF}
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Collecting from {REF} at commit {COMMIT}")

!pip install -q -r {REPO}/requirements.txt

In [ ]:
# Confirm the installed packages match requirements.txt (one test per pin)
!cd {REPO} && python -m pytest phase_1/scripts/test_pinned_versions.py -q -p no:cacheprovider

## Run the collection

Writes to local disk (`/content/phase1_<commit>`): much faster than writing through the Drive mount. Progress and a per-stage timing breakdown are logged as it runs.

In [ ]:
import json
import logging
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# Python reuses modules it has already imported, so a second run in the same
# session would silently use the previous REF's code
stale = [m for m in ("full_pipeline", "forward_pass", "pytorch_forward_hooks", "determinism_validation")
         if m in sys.modules]
assert not stale, (f"Code from an earlier run is still loaded ({', '.join(stale)}). "
                   "Runtime → Restart session, then Run all again.")

sys.path.insert(0, f"{REPO}/phase_1/scripts")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)

import torch
import transformers
from full_pipeline import run_full_pipeline

output_dir = Path(f"/content/phase1_{COMMIT}")
assert not output_dir.exists() or not any(output_dir.iterdir()), (
    f"{output_dir} already has data. Delete it, or collect from a different REF.")
output_dir.mkdir(parents=True, exist_ok=True)

provenance = {
    "ref": REF,
    "commit": COMMIT,
    "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "num_orderings": NUM_ORDERINGS,
    "subset_size": SUBSET_SIZE,
}

start = time.time()
stats = run_full_pipeline(num_runs=NUM_ORDERINGS, subset_size=SUBSET_SIZE, output_dir=output_dir)

provenance.update({
    "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "collection_minutes": round((time.time() - start) / 60, 1),
    "prompt_passes": stats.total_prompts_processed,
    "routing_rows": stats.total_routing_events,
    "full_passes_completed": stats.full_passes_completed,
    "orderings_completed": stats.runs_completed,
    "collection_valid": stats.validation_passed,
    "errors": stats.errors,
    "timings_seconds": {stage: round(s, 1) for stage, s in stats.timings.items()},
})
(output_dir / "provenance.json").write_text(json.dumps(provenance, indent=2))

print(f"\nCollected in {provenance['collection_minutes']} min: {stats.total_prompts_processed:,} prompt-passes, "
      f"{stats.total_routing_events:,} rows")
print(f"Full passes {stats.full_passes_completed}/2, orderings {stats.runs_completed}/{NUM_ORDERINGS}, "
      f"collection valid: {stats.validation_passed}, errors: {stats.errors or 'none'}")

## Validate

**The determinism check is currently expected to fail.** Routing depends slightly on which prompts share a batch: about 2% of top-4 choices differ between orderings, always at near-ties, and the differences cascade into later layers (discussion #114). Until the team decides what the check should require (#105), read the **randomization** line (every top-4 change should be at a near-tie, with 0 unexplained) and the batch-variance analysis below.

In [ ]:
try:
    from determinism_validation import DeterminismValidator
except ModuleNotFoundError:
    print(f"determinism_validation.py isn't in {REF} yet (it comes with #60); skipping validation")
else:
    validator = DeterminismValidator(output_dir)
    report = validator.validate_all()
    validator.save_report(report, output_dir / "validation_report.json")
    print(report.summary)

In [ ]:
# Where top-4 changes happen, by layer and position, and the cascade test (#114)
analysis = Path(REPO) / "phase_1" / "scripts" / "batch_variance.py"
if analysis.exists():
    !python {analysis} {output_dir}/determinism | tee {output_dir}/batch_variance.txt
else:
    print(f"batch_variance.py isn't in {REF} yet; skipping")

In [ ]:
# Everything written, with sizes
total = 0
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        total += f.stat().st_size
        print(f"  {str(f.relative_to(output_dir)):40s} {f.stat().st_size / 1e6:8.1f} MB")
print(f"  {'total':40s} {total / 1e6:8.1f} MB")

## Copy to Drive

Copies the whole output folder to a Drive subfolder named after the commit, for example `moe_phase1_data/phase1_38ef1da/`. About 3.3 GB for a full collection, so it takes a few minutes and uses Drive quota.

In [ ]:
if COPY_TO_DRIVE:
    target = Path(DRIVE_FOLDER) / output_dir.name
    assert not target.exists(), f"{target} already exists on Drive; not overwriting it"
    target.parent.mkdir(parents=True, exist_ok=True)
    !cp -r "{output_dir}" "{target}"
    !du -sh "{target}"
    print(f"Copied to {target}")
else:
    print(f"Not copied. The results are only on this runtime's local disk: {output_dir}")

## Finish

1. Check `provenance.json` and `validation_report.json` were copied.
2. **Runtime → Disconnect and delete runtime**, to stop GPU billing. This wipes the local disk, so copy first.
3. Record the run on the relevant issue: commit, GPU, time, and the validation summary.

**To run something else on this GPU first** (another `REF`, or GPU tests in the terminal): **Runtime → Restart session**. This notebook's kernel keeps the model (about 30 GB) in GPU memory, and keeps the previous run's code loaded.